<a href="https://colab.research.google.com/github/NelvaAdalit/-INTELIGENCIA-ARTIFICIAL-I-/blob/main/PROYECTFINALIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
thomasdubail_chest_pneumonia_256x256_path = kagglehub.dataset_download('thomasdubail/chest-pneumonia-256x256')

print('Data source import complete.')


# 🫁 Diagnóstico de Neumonía — MLP (Red Densa)
**Materia:** Inteligencia Artificial I (SIS420)  
**Mora Barrionuevo Nelva Adalit**


## FASE 1 — Inicialización del Hardware

In [ ]:
# ==============================================================================
# FASE 1: INICIALIZACIÓN DEL HARDWARE
# ==============================================================================
import torch
import torch.nn as nn
import os
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("--- REPORTE DE HARDWARE ---")
print(f"Dispositivo principal: {device}")
print(f"Cantidad de GPUs detectadas: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"  Memoria Total: {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB")
else:
    print("Sin GPU disponible — entrenando en CPU.")

## FASE 2 — Análisis del Dataset y Desbalanceo

In [ ]:
# ==============================================================================
# FASE 2: ANÁLISIS DEL DATASET (El "Antes" del desbalanceo)
# ==============================================================================
base_data_dir = '/kaggle/input/datasets/thomasdubail/chest-pneumonia-256x256/Data'
subsets = ['train', 'val', 'test']
clases = ['normal', 'pneumonia']

stats = {subset: {clase: 0 for clase in clases} for subset in subsets}

for subset in subsets:
    for clase in clases:
        ruta = os.path.join(base_data_dir, subset, clase)
        if os.path.exists(ruta):
            stats[subset][clase] = len(os.listdir(ruta))

print("--- ANÁLISIS ESTADÍSTICO COMPLETO DEL DATASET ---")
for subset in subsets:
    total_subset = sum(stats[subset].values())
    print(f"\nConjunto: {subset.upper()} (Total: {total_subset})")
    for clase in clases:
        pct = stats[subset][clase] / total_subset * 100 if total_subset > 0 else 0
        print(f"  - {clase.capitalize():12s}: {stats[subset][clase]:4d} imágenes ({pct:.1f}%)")

# --- GRÁFICO COMPARATIVO DE DISTRIBUCIÓN ---
x = range(len(subsets))
width = 0.35

plt.figure(figsize=(10, 5))
normal_counts    = [stats[s]['normal']    for s in subsets]
pneumonia_counts = [stats[s]['pneumonia'] for s in subsets]

bars1 = plt.bar(x, normal_counts,    width, label='Normal',   color='#2ca02c', alpha=0.85)
bars2 = plt.bar([i + width for i in x], pneumonia_counts, width, label='Neumonía', color='#d62728', alpha=0.85)

for bar in bars1 + bars2:
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
             str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)

plt.xlabel('Subconjuntos del Dataset')
plt.ylabel('Cantidad de Imágenes')
plt.title('Distribución de Clases por Subset\n(Desbalanceo: Neumonía ≈ 4× más que Normal en Train)')
plt.xticks([i + width/2 for i in x], [s.upper() for s in subsets])
plt.legend()
plt.tight_layout()
plt.show()

print("\n⚠️  Desbalanceo confirmado. Estrategia: Data Augmentation + Dropout")

## FASE 3 — Data Augmentation y DataLoaders
> **Por qué 64×64:** Aplanar una imagen de 256×256 produce un vector de 65.536 valores → millones de parámetros → colapso de RAM. Con 64×64 el vector aplanado es de 4.096 valores, manejable para un MLP en Kaggle (~16 GB RAM).

In [ ]:
# ==============================================================================
# FASE 3: DATA AUGMENTATION Y DATALOADERS
# ==============================================================================
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# --- 1. TRANSFORMACIONES ---
# Entrenamiento: aplica Data Augmentation para mitigar el desbalanceo
train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),   # 1 canal: escala de grises
    transforms.Resize((64, 64)),                   # Reducir para el MLP
    transforms.RandomHorizontalFlip(p=0.5),        # Augmentation: espejo
    transforms.RandomRotation(degrees=15),         # Augmentation: rotación
    transforms.ToTensor()                          # Normaliza a [0, 1]
])

# Validación y Test: SOLO reescalado, sin augmentation
test_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

# --- 2. CARGA DE DATASETS ---
train_dataset = datasets.ImageFolder(os.path.join(base_data_dir, 'train'), transform=train_transforms)
val_dataset   = datasets.ImageFolder(os.path.join(base_data_dir, 'val'),   transform=test_transforms)
test_dataset  = datasets.ImageFolder(os.path.join(base_data_dir, 'test'),  transform=test_transforms)

# --- 3. DATALOADERS (batch_size=8 para evitar Out of Memory en Kaggle) ---
batch_size = 8
dataloader = {
    'train': DataLoader(train_dataset, batch_size=batch_size, shuffle=True),
    'val':   DataLoader(val_dataset,   batch_size=batch_size, shuffle=False),
    'test':  DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)
}

print("--- RESUMEN DE DATALOADERS ---")
print(f"Clases detectadas: {train_dataset.classes}  →  {train_dataset.class_to_idx}")
print(f"Imágenes de entrenamiento : {len(train_dataset):,}  ({len(dataloader['train'])} batches)")
print(f"Imágenes de validación    : {len(val_dataset):,}  ({len(dataloader['val'])} batches)")
print(f"Imágenes de test          : {len(test_dataset):,}  ({len(dataloader['test'])} batches)")
print(f"\nTamaño del tensor por batch: {next(iter(dataloader['train']))[0].shape}")
print("  → [batch=8, canales=1, alto=64, ancho=64] ✓")

# --- 4. VISUALIZACIÓN DE MUESTRA CON DATA AUGMENTATION ---
images, labels = next(iter(dataloader['train']))
clases_nombres = train_dataset.classes  # ['normal', 'pneumonia']

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
fig.suptitle('Muestra del DataLoader — Imágenes con Data Augmentation (64×64, Escala de Grises)', fontsize=13)

for i, ax in enumerate(axes.flat):
    img = images[i].squeeze().numpy()  # quitar canal, pasar a numpy
    etiqueta = clases_nombres[labels[i].item()]
    color_titulo = '#2ca02c' if etiqueta == 'normal' else '#d62728'
    ax.imshow(img, cmap='gray')
    ax.set_title(etiqueta.upper(), color=color_titulo, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()


## FASE 4 — Arquitectura MLP (Perceptrón Multicapa)
>`nn.ReLU` + `nn.Dropout`. El `nn.Flatten()` convierte la matriz 64×64 en un vector de 4.096 elementos.

In [ ]:
# ==============================================================================
# FASE 4: ARQUITECTURA MLP — PERCEPTRÓN MULTICAPA DENSO
# ==============================================================================

class MLP_Neumonia(nn.Module):
    """
    Red Neuronal Densa (Fully Connected) para clasificación binaria.
    Entrada: imagen 64x64 en escala de grises → vector aplanado de 4096 valores.
    Salida:  2 logits (Normal / Neumonía) — CrossEntropyLoss aplica softmax internamente.
    """
    def __init__(self, input_size=64*64, hidden1=512, hidden2=256, num_classes=2, dropout_rate=0.4):
        super().__init__()

        # Aplanado obligatorio: matriz [1,64,64] → vector [4096]
        self.flatten = nn.Flatten()

        # Capa 1: 4096 → 512 | ReLU + Dropout
        self.fc1   = nn.Linear(input_size, hidden1)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(dropout_rate)  # Regularización: apaga 40% neuronas

        # Capa 2: 512 → 256 | ReLU + Dropout
        self.fc2   = nn.Linear(hidden1, hidden2)
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(dropout_rate)

        # Capa de Salida: 256 → 2 (sin activación, CrossEntropyLoss la incluye)
        self.salida = nn.Linear(hidden2, num_classes)

    def forward(self, x):
        x = self.flatten(x)                    # [B, 1, 64, 64] → [B, 4096]
        x = self.drop1(self.relu1(self.fc1(x)))  # [B, 4096] → [B, 512]
        x = self.drop2(self.relu2(self.fc2(x)))  # [B, 512]  → [B, 256]
        return self.salida(x)                  # [B, 256]  → [B, 2]

modelo = MLP_Neumonia().to(device)

# --- REPORTE DE ARQUITECTURA ---
print("--- ARQUITECTURA DEL MODELO (MLP — Solo capas densas) ---")
print(modelo)

total_params = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
print(f"\nTotal de parámetros entrenables: {total_params:,}")
print(f"Dispositivo asignado: {next(modelo.parameters()).device}")
print("\n✓ Sin capas convolucionales (Conv2d). Red 100% densa.")

## FASE 5 — Configuración: Criterio, Optimizador y Scheduler
>  El desbalanceo se maneja con Data Augmentation.  
> **Scheduler `ReduceLROnPlateau`:** Reduce el learning rate a la mitad si el F1 de validación no mejora en 5 épocas.

In [ ]:
# ==============================================================================
# FASE 5: CRITERIO, OPTIMIZADOR Y SCHEDULER
# ==============================================================================

# Instalamos torchmetrics silenciando los errores de dependencias nativas de Kaggle
!pip install torchmetrics -q > /dev/null 2>&1

import torch
import torch.nn as nn
from torchmetrics.classification import MulticlassAccuracy, MulticlassF1Score, MulticlassRecall

# --- CRITERIO: CrossEntropyLoss estándar
criterio = nn.CrossEntropyLoss()

# --- OPTIMIZADOR: Adam con weight_decay = regularización L2 ---
optimizador = torch.optim.Adam(
    modelo.parameters(),
    lr=0.0005,          # Learning rate inicial
    weight_decay=1e-4   # L2: penaliza pesos grandes para evitar sobreajuste
)

# --- SCHEDULER: Reduce LR cuando el F1 de validación no mejora ---
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizador,
    mode='max',      # Queremos MAXIMIZAR el F1
    factor=0.5,      # Multiplica LR × 0.5 cuando dispara
    patience=5       # Espera 5 épocas sin mejora antes de reducir
)

# --- MÉTRICAS (torchmetrics) ---
metric_f1     = MulticlassF1Score(num_classes=2,  average='macro').to(device)
metric_acc    = MulticlassAccuracy(num_classes=2, average='macro').to(device)
metric_recall = MulticlassRecall(num_classes=2,   average='macro').to(device)

print("--- CONFIGURACIÓN DEL ENTRENAMIENTO ---")
print(f"Criterio   : CrossEntropyLoss ")
print(f"Optimizador: Adam  |  lr=0.0005  |  weight_decay=1e-4")
print(f"Scheduler  : ReduceLROnPlateau  |  mode=max  |  factor=0.5  |  patience=5")
print(f"Métricas   : F1-Score (macro), Accuracy (macro), Recall (macro)")
print(f"Dispositivo: {device}")


## FASE 6 — La Receta de Entrenamiento (Early Stopping + Checkpoints)
>  El Scheduler ajusta el learning rate. El Early Stopping detiene si no hay mejora en 15 épocas.

In [ ]:
# ==============================================================================
# FASE 6: LA RECETA DE ENTRENAMIENTO
# ==============================================================================

def fit(model, dataloader, optimizer, criterion, scheduler, epochs=96, patience=15):
    """
    Bucle de entrenamiento completo con:
      - Checkpoint automático del mejor modelo (.pth)
      - Early Stopping por F1 de validación
      - Scheduler que reduce LR cuando el F1 se estanca
      - Historial completo de métricas para graficar
    """
    best_f1 = 0.0
    epochs_sin_mejora = 0
    historial = {
        'loss_train': [], 'loss_val': [],
        'f1_val': [], 'acc_val': [], 'recall_val': []
    }

    print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>8} | {'F1 Val':>7} | {'Acc Val':>7} | LR actual")
    print("-" * 72)

    for epoch in range(1, epochs + 1):

        # ── ENTRENAMIENTO ──────────────────────────────────────────────────────
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in dataloader['train']:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(dataloader['train'])

        # ── VALIDACIÓN ─────────────────────────────────────────────────────────
        model.eval()
        val_loss = 0.0
        metric_f1.reset()
        metric_acc.reset()
        metric_recall.reset()

        with torch.no_grad():
            for X_batch, y_batch in dataloader['val']:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                y_pred = model(X_batch)
                val_loss += criterion(y_pred, y_batch).item()
                metric_f1.update(y_pred, y_batch)
                metric_acc.update(y_pred, y_batch)
                metric_recall.update(y_pred, y_batch)

        val_loss   /= len(dataloader['val'])
        val_f1      = metric_f1.compute().item()
        val_acc     = metric_acc.compute().item()
        val_recall  = metric_recall.compute().item()

        # Guardar historial
        historial['loss_train'].append(train_loss)
        historial['loss_val'].append(val_loss)
        historial['f1_val'].append(val_f1)
        historial['acc_val'].append(val_acc)
        historial['recall_val'].append(val_recall)

        # Scheduler: ajusta LR según F1 de validación
        scheduler.step(val_f1)
        lr_actual = optimizer.param_groups[0]['lr']

        # ── CHECKPOINT ────────────────────────────────────────────────────────
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), 'mejor_mlp_neumonia.pth')
            epochs_sin_mejora = 0
            marca = "⭐ CHECKPOINT GUARDADO"
        else:
            epochs_sin_mejora += 1
            marca = ""

        # Imprimir cada época
        print(f"{epoch:>6} | {train_loss:>10.4f} | {val_loss:>8.4f} | {val_f1:>7.4f} | {val_acc:>7.4f} | {lr_actual:.6f}  {marca}")

        # ── EARLY STOPPING ───────────────────────────────────────────────────
        if epochs_sin_mejora >= patience:
            print(f"\n🛑 Early Stopping activado en la época {epoch}.")
            print(f"   El modelo no mejoró en {patience} épocas consecutivas.")
            break

    # Cargar los mejores pesos antes de devolver
    model.load_state_dict(torch.load('mejor_mlp_neumonia.pth'))
    print(f"\n✓ Entrenamiento finalizado. Mejor F1-Val: {best_f1:.4f}")
    print("  Pesos óptimos cargados desde 'mejor_mlp_neumonia.pth'")
    return historial


# ── LANZAR ENTRENAMIENTO ──────────────────────────────────────────────────────
print(f"Iniciando entrenamiento: hasta 96 épocas | Early Stopping: patience=15\n")
historial = fit(
    modelo, dataloader, optimizador, criterio, scheduler,
    epochs=96, patience=15
)

## FASE 7 — Curvas de Aprendizaje

In [ ]:
# ==============================================================================
# FASE 7: CURVAS DE APRENDIZAJE
# ==============================================================================
epocas = range(1, len(historial['loss_train']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Curvas de Aprendizaje — MLP Neumonía', fontsize=14, fontweight='bold')

# Gráfica 1: Loss Train vs Val
axes[0].plot(epocas, historial['loss_train'], label='Train Loss', color='#1f77b4')
axes[0].plot(epocas, historial['loss_val'],   label='Val Loss',   color='#ff7f0e')
axes[0].set_title('Función de Pérdida (Loss)')
axes[0].set_xlabel('Épocas')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Gráfica 2: F1-Score y Accuracy
axes[1].plot(epocas, historial['f1_val'],  label='F1-Score (Val)',  color='#2ca02c')
axes[1].plot(epocas, historial['acc_val'], label='Accuracy (Val)', color='#9467bd')
axes[1].set_title('F1-Score y Accuracy (Validación)')
axes[1].set_xlabel('Épocas')
axes[1].set_ylabel('Puntaje')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Gráfica 3: Recall
axes[2].plot(epocas, historial['recall_val'], label='Recall (Val)', color='#d62728')
axes[2].set_title('Recall (Validación)')
axes[2].set_xlabel('Épocas')
axes[2].set_ylabel('Recall')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Épocas entrenadas     : {len(epocas)}")
print(f"Mejor F1-Val          : {max(historial['f1_val']):.4f}")
print(f"Mejor Accuracy-Val    : {max(historial['acc_val']):.4f}")
print(f"Mejor Recall-Val      : {max(historial['recall_val']):.4f}")

## FASE 8 — Evaluación Final en Test + Matriz de Confusión
> Se carga el mejor `.pth` guardado. El modelo entra en modo `eval()` (Dropout desactivado). Se evalúa en datos **nunca vistos** durante el entrenamiento.

In [ ]:
# ==============================================================================
# FASE 8: EVALUACIÓN FINAL EN TEST Y MATRIZ DE CONFUSIÓN
# ==============================================================================
from torchmetrics.classification import MulticlassConfusionMatrix
from sklearn.metrics import classification_report
import seaborn as sns

# --- 1. CARGAR EL MEJOR MODELO GUARDADO ---
modelo_test = MLP_Neumonia().to(device)
modelo_test.load_state_dict(torch.load('mejor_mlp_neumonia.pth', map_location=device))
modelo_test.eval()  # ¡CRÍTICO: desactiva Dropout para inferencia!

# --- 2. MÉTRICAS DE TEST ---
metric_acc_test    = MulticlassAccuracy(num_classes=2,  average='macro').to(device)
metric_f1_test     = MulticlassF1Score(num_classes=2,   average='macro').to(device)
metric_recall_test = MulticlassRecall(num_classes=2,    average='macro').to(device)
conf_matrix        = MulticlassConfusionMatrix(num_classes=2).to(device)

y_true_all = []
y_pred_all = []

with torch.no_grad():
    for X_batch, y_batch in dataloader['test']:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        y_pred = modelo_test(X_batch)

        metric_acc_test.update(y_pred, y_batch)
        metric_f1_test.update(y_pred, y_batch)
        metric_recall_test.update(y_pred, y_batch)
        conf_matrix.update(y_pred, y_batch)

        y_true_all.extend(y_batch.cpu().numpy())
        y_pred_all.extend(torch.argmax(y_pred, dim=1).cpu().numpy())

# --- 3. RESULTADOS ---
test_acc    = metric_acc_test.compute().item()
test_f1     = metric_f1_test.compute().item()
test_recall = metric_recall_test.compute().item()

print("═" * 45)
print("   RESULTADOS FINALES — CONJUNTO DE TEST")
print("   (Datos nunca vistos durante el entrenamiento)")
print("═" * 45)
print(f"  Accuracy (Macro) : {test_acc * 100:.2f}%")
print(f"  F1-Score (Macro) : {test_f1 * 100:.2f}%")
print(f"  Recall   (Macro) : {test_recall * 100:.2f}%")
print("═" * 45)

# Reporte por clase
print("\n--- REPORTE DETALLADO POR CLASE ---")
print(classification_report(y_true_all, y_pred_all, target_names=['Normal', 'Neumonía']))

# --- 4. MATRIZ DE CONFUSIÓN ---
matriz_np = conf_matrix.compute().cpu().numpy()

plt.figure(figsize=(7, 5))
sns.heatmap(
    matriz_np, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Normal', 'Neumonía'],
    yticklabels=['Normal', 'Neumonía']
)
plt.xlabel('Predicción del Modelo', fontsize=12)
plt.ylabel('Diagnóstico Real',       fontsize=12)
plt.title('Matriz de Confusión — Conjunto de Test\n(MLP Denso | 64×64 | ', fontsize=13)
plt.tight_layout()
plt.show()

print(f"\nVerdaderos Negativos (TN - Normal correcto)    : {int(matriz_np[0][0])}")
print(f"Falsos Positivos     (FP - Normal → Neumonía)  : {int(matriz_np[0][1])}")
print(f"Falsos Negativos     (FN - Neumonía → Normal)  : {int(matriz_np[1][0])}")
print(f"Verdaderos Positivos (TP - Neumonía correcto)  : {int(matriz_np[1][1])}")

## FASE 9 — Inferencia Interactiva en Kaggle (ipywidgets)
> Interfaz para subir una radiografía y obtener diagnóstico en tiempo real.

In [ ]:
# ==============================================================================
# FASE 9: INFERENCIA INTERACTIVA (ipywidgets)
# ==============================================================================
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
import io

# Transformaciones para inferencia (iguales a test, sin augmentation)
inferencia_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

def predecir_imagen(boton):
    with salida_UI:
        clear_output()
        if not boton_subir.value:
            print("⚠️  Por favor, sube una imagen de radiografía primero.")
            return

        # Compatibilidad con la API de widgets en Kaggle
        uploaded_data = boton_subir.value
        content = (
            uploaded_data[0]['content']
            if isinstance(uploaded_data, tuple)
            else list(uploaded_data.values())[0]['content']
        )

        # Procesar imagen
        imagen_pil = Image.open(io.BytesIO(content)).convert('L')
        tensor_img = inferencia_transforms(imagen_pil).unsqueeze(0).to(device)

        # Inferencia con el mejor modelo
        modelo_test.eval()
        with torch.no_grad():
            output = modelo_test(tensor_img)
            probs  = torch.nn.functional.softmax(output, dim=1)
            clase_idx  = torch.argmax(probs, dim=1).item()
            confianza  = probs[0][clase_idx].item() * 100

        clases     = ['NORMAL', 'NEUMONÍA']
        resultado  = clases[clase_idx]
        color      = 'green' if clase_idx == 0 else 'red'

        display(imagen_pil.resize((256, 256)))
        html_resultado = f"""
        <div style="background:#f9f9f9; padding:20px; border-radius:10px;
                    border:3px solid {color}; text-align:center; margin-top:15px;">
            <h2 style="color:{color}; font-size:26px; margin:0;">DIAGNÓSTICO: {resultado}</h2>
            <p style="font-size:18px; margin-top:8px;">Confianza de la red: <b>{confianza:.2f}%</b></p>
        </div>
        """
        display(widgets.HTML(html_resultado))

# --- INTERFAZ ---
boton_subir    = widgets.FileUpload(accept='image/*', multiple=False, description='Subir Rayos-X')
boton_analizar = widgets.Button(description='Analizar Imagen', button_style='success', icon='stethoscope')
salida_UI      = widgets.Output()

boton_analizar.on_click(predecir_imagen)

display(widgets.VBox([
    widgets.HTML("<h2 style='color:#333;'>🩺 Asistente de Diagnóstico Clínico — MLP Neumonía</h2>"
                 "<p>Sube una imagen de Rayos-X frontal de tórax y presiona 'Analizar'.</p>"),
    widgets.HBox([boton_subir, boton_analizar]),
    salida_UI
]))

In [ ]:
from IPython.display import FileLink
FileLink('mejor_mlp_neumonia.pth')